<a href="https://colab.research.google.com/github/yjin502-cmyk/Infosys722/blob/main/INFOSYS_722_Iteration_3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFOSYS 722 Iteration 3 — report-order Colab notebook

本笔记本按报告 1.1→8.5 顺序排列。逐格从上至下运行；真正的 Colab 运行截图应从运行后的界面截取。A、B 为完整基线和修订轮；完成其评估后，C、D 在 §8.4 再次改变步骤、重跑全套候选模型。D 的 93 条记录属于非随机子样本，不能代替主分析。

## 运行准备 / Runtime setup

将整个 `INFOSYS_722_Iteration_3_Project.zip` 上传到 Colab 左侧 Files，然后运行本格。如果找不到 ZIP，弹出的上传窗口可以选择它。

In [ ]:
from pathlib import Path
import zipfile
project_root = Path.cwd()
data_dir = project_root / 'data'
data_dir.mkdir(exist_ok=True)
required = ('Global_Education.csv', 'WorldbankGDP.csv')
if not all((data_dir / name).is_file() for name in required):
    candidates = (sorted(project_root.glob('INFOSYS_722_Iteration_3_Project*.zip')) +
                  sorted(Path('/').glob('INFOSYS_722_Iteration_3_Project*.zip')))
    if candidates:
        archive_path = candidates[0]
        print('Using uploaded ZIP:', archive_path)
    else:
        from google.colab import files
        uploaded = files.upload()
        archive_name = next((name for name in uploaded if name.endswith('.zip')), None)
        if archive_name is None:
            raise ValueError('Upload INFOSYS_722_Iteration_3_Project.zip')
        archive_path = project_root / archive_name
    with zipfile.ZipFile(archive_path) as archive:
        for filename in required:
            matches = [name for name in archive.namelist()
                       if name.endswith('/data/' + filename)]
            if len(matches) != 1:
                raise ValueError(f'Expected exactly one {filename} in the ZIP')
            (data_dir / filename).write_bytes(archive.read(matches[0]))
print('Data ready:', [name for name in required])

### 安装依赖 / Environment

若首次安装导致自动重启，连接恢复后再次点击 Runtime → Run all。

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
pins = {'numpy':'2.3.5','pandas':'2.2.3','scipy':'1.17.0',
        'scikit-learn':'1.8.0','matplotlib':'3.10.8','Pillow':'12.3.0'}
def current(name):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None
changes = {name: version for name, version in pins.items()
           if current(name) != version}
if changes:
    print('Installing:', changes, flush=True)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
                           '--disable-pip-version-check',
                           *(name+'=='+version for name,version in pins.items())])
    print('Runtime will restart. Then click Runtime → Run all again.', flush=True)
    get_ipython().kernel.do_shutdown(restart=True)
else:
    probe = subprocess.run([sys.executable,'-c',
        'import numpy, numpy.testing, scipy, sklearn, pandas, matplotlib'],
        capture_output=True, text=True)
    if probe.returncode:
        marker = project_root / '.iteration3_environment_restart_attempted'
        if marker.exists():
            raise RuntimeError('Import still fails; disconnect and delete the Colab runtime, '
                               'then reconnect and rerun.\n'+probe.stderr[-1200:])
        marker.touch()
        print('Restarting conflicting runtime once; rerun all after reconnect.', flush=True)
        get_ipython().kernel.do_shutdown(restart=True)
    else:
        (project_root / '.iteration3_environment_restart_attempted').unlink(missing_ok=True)
        print('Dependencies verified.')

In [ ]:
"""INFOSYS 722 Iteration 3: four documented open-source modelling passes.

Run from this folder with: python run_analysis.py
All input files remain unchanged in data/. Generated tables and figures go to outputs/.
"""

from __future__ import annotations

import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler


ROOT = Path.cwd()
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
FIG = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 120})

# These exclusions and the ten World Bank recodes reproduce the decisions
# documented in the archived SPSS stream, rather than infer new country aliases.
EXCLUDED_COUNTRIES = {
    'Anguilla', 'British Virgin Islands', 'Cook Islands', 'Vatican City',
    'Montserrat', 'Nauru', 'Niue', 'Tokelau', 'Turks and Caicos Islands', 'Tuvalu',
}
WB_TO_EDUCATION = {
    'Russian Federation': 'Russia',
    'Viet Nam': 'Vietnam',
    'Czechia': 'Czech Republic',
    "Cote d'Ivoire": 'Ivory Coast',
    'Congo, Rep.': 'Republic of the Congo',
    'Korea, Rep.': 'South Korea',
    'Yemen, Rep.': 'Yemen',
    'Bahamas, The': 'The Bahamas',
    'Cabo Verde': 'Cape Verde',
    'Congo, Dem. Rep.': 'Democratic Republic of the Congo',
}
ID = 'Countries and areas'
PROFILE_FIELDS = [
    'OOSR_Primary_Age_Female', 'OOSR_Upper_Secondary_Age_Female',
    'Completion_Rate_Primary_Female', 'Completion_Rate_Upper_Secondary_Female',
    'Youth_15_24_Literacy_Rate_Female',
    'Gross_Primary_Education_Enrollment',
    'Gross_Tertiary_Education_Enrollment', 'Birth_Rate',
    'Unemployment_Rate', 'GDP_per_capita_2024',
]



from IPython.display import display, Image

## 1.1 Business Objectives

SDG 4 education profiles support further review, not policy ranking.

## 1.2 Situation Assessment

Archive two CSVs, record zero-code and country-join risks.

## 1.3 Data Mining Goals

Inspect groups, separation, coverage, and instability.

## 1.4 Day to Day Project Plan

See dated plan in the report; these cells show executed steps.

## 2.1 Initial Data Collection — read the two archived CSVs

In [ ]:
edu = pd.read_csv(DATA / 'Global_Education.csv', encoding='latin1')
gdp = pd.read_csv(DATA / 'WorldbankGDP.csv', encoding='latin1')
gdp.columns = gdp.columns.str.replace('ï»¿', '', regex=False).str.lstrip('\ufeff')
assert edu.shape == (202, 29) and gdp.shape == (265, 70)
print('Education',edu.shape,'GDP',gdp.shape)

## 2.2 Format Quantity and Fields — inspect the raw data

In [ ]:
display(edu.head(3)); print(edu.dtypes); print('GDP 2024 non-null',gdp['2024'].notna().sum())

## 2.3 Explore the Raw Data — original 202 rows

In [ ]:
def save_figure(name: str):
    plt.tight_layout()
    plt.savefig(FIG / name, dpi=180, bbox_inches='tight')
    plt.close()

def plot_raw(edu: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    fields = [
        ('Completion_Rate_Primary_Female', 'Female primary completion'),
        ('OOSR_Upper_Secondary_Age_Female', 'Female upper-secondary out-of-school'),
    ]
    for ax, (field, label) in zip(axes, fields):
        ax.hist(edu[field], bins=np.arange(-0.5, 101.5, 5), color='#37618d',
                edgecolor='white')
        ax.set(xlabel=label + ' (%)', ylabel='Countries and areas')
        ax.text(0.97, 0.95, f'Zero: {(edu[field] == 0).sum()} / 202',
                ha='right', va='top', transform=ax.transAxes)
    save_figure('raw_distributions.png')

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    for ax, x, y, xlabel, ylabel in [
        (axes[0], 'Completion_Rate_Upper_Secondary_Male',
         'Completion_Rate_Upper_Secondary_Female', 'Male upper-secondary completion (%)',
         'Female upper-secondary completion (%)'),
        (axes[1], 'Birth_Rate', 'Gross_Tertiary_Education_Enrollment',
         'Birth rate', 'Gross tertiary enrolment (%)'),
    ]:
        ax.scatter(edu[x], edu[y], s=19, alpha=0.65, color='#37618d')
        ax.set(xlabel=xlabel, ylabel=ylabel)
        ax.text(0.03, 0.95, f'Pearson r = {edu[x].corr(edu[y]):.3f}',
                va='top', transform=ax.transAxes)
    save_figure('raw_relationships.png')

plot_raw(edu)
for filename in ('raw_distributions.png','raw_relationships.png'):
    display(Image(filename=str(FIG/filename),width=850))

## 2.4 Verify Data Quality — nulls, duplicates, zeros

In [ ]:
quality = pd.DataFrame({
    'field': edu.select_dtypes(include='number').columns,
    'zero_count_raw': edu.select_dtypes(include='number').eq(0).sum().to_numpy(),
})
quality['zero_pct_raw'] = 100 * quality['zero_count_raw'] / len(edu)
quality.to_csv(OUT / 'raw_zero_audit.csv', index=False)
print('Null cells',int(edu.isna().sum().sum()),'duplicate rows',int(edu.duplicated().sum()))
display(quality.sort_values('zero_pct_raw',ascending=False).head(10))

## 3.1 Select Data — repeat ten prior exclusions

In [ ]:
selected = edu.loc[~edu[ID].isin(EXCLUDED_COUNTRIES)].copy()
assert len(selected) == 192
print('After explicit exclusions',len(selected))

## 3.2 Clean Data — preserve raw labels and audit corrections

In [ ]:
selected['Country_Clean'] = selected[ID].replace({
    'Sï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿': 'Sao Tome and Principe',
})
display(selected.loc[selected[ID] != selected.Country_Clean,[ID,'Country_Clean']])

## 3.3 Construct Data — derive completion differences

In [ ]:
selected['Primary_Completion_Gender_Gap'] = (
    selected['Completion_Rate_Primary_Female'] -
    selected['Completion_Rate_Primary_Male'])
selected['UpperSec_Completion_Gender_Gap'] = (
    selected['Completion_Rate_Upper_Secondary_Female'] -
    selected['Completion_Rate_Upper_Secondary_Male'])
display(selected[[ID,'Primary_Completion_Gender_Gap','UpperSec_Completion_Gender_Gap']].head())

## 3.4 Integrate Data — 192→171 matched, then five without GDP

In [ ]:
selected['Merge_Country'] = selected[ID]
gdp['Merge_Country'] = gdp['Country Name'].replace(WB_TO_EDUCATION)
# Deliberately do not infer extra aliases. In particular, the corrupted
# Sao Tome label and Guinea0Bissau do not match in the archived SPSS flow.
joined = selected.merge(
    gdp[['Merge_Country', 'Country Name', 'Country Code', '2024']],
    on='Merge_Country', how='inner', validate='one_to_one',
)
assert len(joined) == 171
join_audit = selected[[ID, 'Merge_Country', 'Country_Clean']].merge(
    gdp[['Merge_Country', 'Country Name', '2024']],
    on='Merge_Country', how='left', indicator=True, validate='one_to_one')
join_audit['disposition'] = np.select([
    join_audit['_merge'].eq('left_only'),
    join_audit['2024'].isna(),
    join_audit['2024'].le(0),
], ['unmatched_name', 'missing_2024_GDP', 'nonpositive_2024_GDP'],
   default='included')
assert join_audit['disposition'].value_counts().to_dict() == {
    'included': 166, 'unmatched_name': 21, 'missing_2024_GDP': 5}
join_audit.sort_values(ID).to_csv(OUT / 'country_join_audit.csv', index=False)
missing_gdp = joined.loc[joined['2024'].isna(), ID].sort_values().tolist()
cohort = joined.loc[joined['2024'].notna() & joined['2024'].gt(0)].copy()
assert len(cohort) == 166 and len(missing_gdp) == 5
display(join_audit['disposition'].value_counts()); display(join_audit.loc[join_audit.disposition!='included',[ID,'disposition']].head(10))

## 3.5 Reformat Data — 166-country modelling cohort

In [ ]:
cohort = cohort.sort_values(ID).reset_index(drop=True)
cohort = cohort.rename(columns={'2024': 'GDP_per_capita_2024'})
cohort['log_GDP_2024'] = np.log(cohort['GDP_per_capita_2024'])
cohort.to_csv(OUT / 'modelling_cohort_166.csv', index=False)

audit = {
    'raw_education_rows': len(edu),
    'raw_education_columns': len(edu.columns),
    'raw_gdp_rows': len(gdp),
    'raw_gdp_columns': len(gdp.columns) - 1,  # excludes the added join field
    'gdp_2024_non_null_raw': int(gdp['2024'].notna().sum()),
    'raw_education_null_cells': int(edu.isna().sum().sum()),
    'raw_education_duplicate_rows': int(edu.duplicated().sum()),
    'raw_education_duplicate_country_labels': int(edu[ID].duplicated().sum()),
    'excluded_education_rows': len(edu) - len(selected),
    'selected_education_rows': len(selected),
    'joined_rows_before_gdp_filter': len(joined),
    'matched_rows_without_2024_gdp': len(missing_gdp),
    'matched_rows_without_2024_gdp_names': missing_gdp,
    'final_cohort_rows': len(cohort),
    'raw_gender_completion_pearson': round(float(edu[
        'Completion_Rate_Upper_Secondary_Male'].corr(
        edu['Completion_Rate_Upper_Secondary_Female'])), 6),
    'raw_birth_tertiary_pearson': round(float(edu['Birth_Rate'].corr(
        edu['Gross_Tertiary_Education_Enrollment'])), 6),
}
print('Final country count',len(cohort));display(cohort[[ID,'GDP_per_capita_2024','log_GDP_2024']].head())

## 4.1 Feature Selection and Scaling — explicit 27→23 decision

In [ ]:
education_fields = [c for c in edu.select_dtypes('number').columns
                    if c not in ('Latitude ', 'Longitude')]
baseline_columns = education_fields + ['log_GDP_2024']
zero_rates = cohort[education_fields].eq(0).mean().sort_values(ascending=False)
excluded_fields = zero_rates[zero_rates > .60].index.tolist()
revision_columns = [f for f in baseline_columns if f not in excluded_fields]
assert len(baseline_columns) == 27 and len(excluded_fields) == 4
assert len(revision_columns) == 23
zero_rates.rename('zero_rate').to_csv(OUT / 'cohort_166_feature_zero_rates.csv')
pd.DataFrame({
    'field': education_fields,
    'zero_share_pct_166': [round(100 * float(zero_rates[f]), 2)
                           for f in education_fields],
    'selected_A': True,
    'selected_B': [f not in excluded_fields for f in education_fields],
    'selected_C_no_GDP': [f not in excluded_fields for f in education_fields],
    'selected_D_n93': [f not in excluded_fields for f in education_fields],
}).to_csv(OUT / 'feature_selection_decisions.csv', index=False)
(OUT / 'revision_excluded_features.json').write_text(json.dumps({
    field: round(100 * float(zero_rates[field]), 2) for field in excluded_fields
}, indent=2))
display(pd.read_csv(OUT/'feature_selection_decisions.csv').query('selected_B == False'))

## 4.2 Principal Components — complete A/B transformation

In [ ]:
def prepare_cycle(cohort: pd.DataFrame, columns: list[str], name: str,
                  fixed_components: int | None = None) -> dict:
    x = StandardScaler().fit_transform(cohort[columns])
    full = PCA(svd_solver='full').fit(x)
    n = fixed_components or int(np.searchsorted(
        full.explained_variance_ratio_.cumsum(), 0.90) + 1)
    fitted = PCA(n_components=n, svd_solver='full').fit(x)
    scores = fitted.transform(x)
    pd.DataFrame({
        'component': np.arange(1, len(full.explained_variance_ratio_) + 1),
        'individual_variance_ratio': full.explained_variance_ratio_,
        'cumulative_variance_ratio': full.explained_variance_ratio_.cumsum(),
    }).to_csv(OUT / f'{name}_pca_variance.csv', index=False)
    pd.DataFrame(fitted.components_.T, index=columns,
                 columns=[f'PC{i+1}' for i in range(n)]).to_csv(
        OUT / f'{name}_pca_loadings.csv', index_label='field')
    return {'name': name, 'columns': columns, 'cohort': cohort, 'scaled': x, 'pca': fitted,
            'full_pca': full, 'scores': scores, 'n_components': n,
            'explained': float(fitted.explained_variance_ratio_.sum())}

def plot_pca(cycles):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
    for ax, cycle in zip(axes, cycles):
        vals = cycle['full_pca'].explained_variance_ratio_.cumsum()
        ax.plot(np.arange(1, len(vals) + 1), 100 * vals, marker='o', markersize=3,
                color='#37618d')
        ax.axhline(90, color='#ae5e3b', linestyle='--', label='90% threshold')
        ax.axvline(cycle['n_components'], color='#697e92', linestyle=':')
        ax.scatter([cycle['n_components']], [100 * cycle['explained']],
                   color='#ae5e3b', zorder=3)
        ax.set(xlabel='Number of components', ylabel='Cumulative variance (%)',
               title=f'{cycle["name"].title()}: {len(cycle["columns"])} inputs, '
                     f'{cycle["n_components"]} PCs ({cycle["explained"]:.1%})',
               xlim=(1, len(vals)), ylim=(25, 101))
    axes[1].legend(loc='lower right')
    save_figure('pca_comparison.png')

    fig, axes = plt.subplots(2, 2, figsize=(10, 6.4), sharey=True)
    for ax, cycle in zip(axes.flat, cycles):
        vals = cycle['full_pca'].explained_variance_ratio_.cumsum()
        ax.plot(np.arange(1, len(vals) + 1), 100 * vals,
                marker='o', markersize=3, color='#37618d')
        ax.axhline(90, color='#ae5e3b', linestyle='--')
        ax.axvline(cycle['n_components'], color='#697e92', linestyle=':')
        ax.set(title=f"{cycle['name']}: n={len(cycle['cohort'])}, "
                     f"{len(cycle['columns'])} inputs, {cycle['n_components']} PCs "
                     f"({cycle['explained']:.1%})",
               xlabel='Number of components', ylabel='Cumulative variance (%)',
               xlim=(1, len(vals)), ylim=(25, 101))
    save_figure('four_pass_pca.png')

cycles = [prepare_cycle(cohort, baseline_columns, 'baseline', 8),
          prepare_cycle(cohort, revision_columns, 'revision')]
assert cycles[1]['n_components'] == 7
plot_pca(cycles);display(Image(filename=str(FIG/'pca_comparison.png'),width=850))

## 5.1 Match Methods to Objectives

Country profiles have no verified target labels; exploratory clustering is suitable.

## 5.2 Select Data Mining Method

Cluster assessment combines fit, size, raw-unit meaning and data-quality checks.

## 6.1 Explore Algorithms

Compare centroid-based K-Means with Ward hierarchical clustering.

## 6.2 Select Algorithms

Fit both algorithms for every k in each pass.

## 6.3 Build Models and Set Parameters — inspect actual settings

In [ ]:
def fit_candidate(scores: np.ndarray, method: str, k: int, seed: int = 42):
    if method == 'KMeans':
        estimator = KMeans(n_clusters=k, random_state=seed, n_init=20,
                           algorithm='lloyd')
    else:
        estimator = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = estimator.fit_predict(scores)
    return estimator, labels

print('K-Means n_init=20, seed=42, Lloyd; Ward linkage; k=2..8')

## 7.1 Test Design and Execution — fit A/B first (28 candidates)

In [ ]:
def compare_models(cycles: list[dict], cohort: pd.DataFrame):
    rows, labels_by_key = [], {}
    assignments = cohort[[ID]].copy()
    for cycle in cycles:
        for method in ('KMeans', 'Ward'):
            for k in range(2, 9):
                model, labels = fit_candidate(cycle['scores'], method, k)
                key = (cycle['name'], method, k)
                labels_by_key[key] = labels
                sizes = np.bincount(labels, minlength=k)
                rows.append({
                    'cycle': cycle['name'], 'method': method, 'k': k,
                    'silhouette': silhouette_score(cycle['scores'], labels),
                    'davies_bouldin': davies_bouldin_score(cycle['scores'], labels),
                    'calinski_harabasz': calinski_harabasz_score(cycle['scores'], labels),
                    'inertia': model.inertia_ if method == 'KMeans' else np.nan,
                    'smallest_cluster': int(sizes.min()),
                    'largest_cluster': int(sizes.max()),
                    'cluster_sizes': '/'.join(str(n) for n in sorted(sizes)),
                })
                # Pass D uses a smaller, selected cohort: explicitly leave the
                # other countries unassigned instead of inventing memberships.
                assignments[f'{cycle["name"]}_{method}_k{k}'] = assignments[ID].map(
                    pd.Series(labels + 1, index=cycle['cohort'][ID]).to_dict())
    comparison = pd.DataFrame(rows)
    comparison.to_csv(OUT / 'model_comparison.csv', index=False,
                      float_format='%.6f')
    assignments.to_csv(OUT / 'all_country_assignments.csv', index=False)
    return comparison, labels_by_key


comparison, labels = compare_models(cycles, cohort)
assert len(comparison)==28
print('A/B candidates',len(comparison),'full cohort',len(cohort))

## 7.2 Candidate Outputs and Selection — statistics of A/B

In [ ]:
def plot_comparison(comparison):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.1))
    colors = {'baseline': '#37618d', 'revision': '#c65d3c',
              'education_only': '#549277', 'completion_nonzero': '#87548a'}
    for (cycle, method), data in comparison.groupby(['cycle', 'method']):
        color = colors[cycle]
        linestyle = '-' if method == 'KMeans' else '--'
        for ax, metric in zip(axes, ('silhouette', 'davies_bouldin')):
            ax.plot(data['k'], data[metric], marker='o', color=color,
                    linestyle=linestyle, label=f'{cycle.title()} {method}')
    axes[0].set(xlabel='Number of clusters k', ylabel='Silhouette (higher is better)')
    axes[1].set(xlabel='Number of clusters k', ylabel='Davies–Bouldin (lower is better)')
    axes[0].legend(fontsize=7, loc='best', ncol=2)
    save_figure('model_metrics.png')

def plot_four_pass_summary(comparison):
    """Show the four completed runs without implying cross-cohort score equivalence."""
    chosen = [('baseline', 5), ('revision', 3),
              ('education_only', 3), ('completion_nonzero', 2)]
    rows = [comparison.loc[(comparison.cycle == name) &
                           (comparison.method == 'KMeans') &
                           (comparison.k == k)].iloc[0] for name, k in chosen]
    colors = ['#37618d', '#c65d3c', '#549277', '#87548a']
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    names = ['A: 166 / k5', 'B: 166 / k3', 'C: 166 / k3', 'D: 93 / k2']
    for ax, metric, label in zip(axes, ['silhouette', 'davies_bouldin'],
                                 ['Silhouette ↑', 'Davies–Bouldin ↓']):
        values = [row[metric] for row in rows]
        bars = ax.bar(names, values, color=colors)
        ax.bar_label(bars, fmt='%.3f', fontsize=8)
        ax.set(ylabel=label, ylim=(0, max(values) * 1.18))
        ax.tick_params(axis='x', labelrotation=12)
    fig.suptitle('Four fitted passes; D selects a different 93-country cohort')
    save_figure('four_pass_summary.png')

plot_comparison(comparison)
display(comparison.loc[(comparison.method=='KMeans') &
 ((comparison.cycle=='baseline')&(comparison.k==5) |
  (comparison.cycle=='revision')&(comparison.k==3))])
display(Image(filename=str(FIG/'model_metrics.png'),width=850))

## 7.3 Search for Statistical Patterns — fitted model scores and country medians

In [ ]:
def profile(cohort: pd.DataFrame, cycle: dict, labels: np.ndarray,
            filename_prefix: str):
    df = cohort[[ID] + PROFILE_FIELDS].copy()
    df['cluster'] = labels + 1  # Labels are arbitrary identifiers, not rankings.
    medians = df.groupby('cluster')[PROFILE_FIELDS].median().round(2)
    sizes = df.groupby('cluster').size().rename('n')
    medians.insert(0, 'n', sizes)
    medians.to_csv(OUT / f'{filename_prefix}_medians.csv')
    zero_share = df.groupby('cluster')[PROFILE_FIELDS[:-1]].apply(
        lambda group: 100 * group.eq(0).mean()).round(1)
    zero_share.to_csv(OUT / f'{filename_prefix}_zero_share_pct.csv')
    full_medians = cohort.select_dtypes('number').copy()
    full_medians['cluster'] = labels + 1
    full_medians.groupby('cluster').median().round(2).to_csv(
        OUT / f'{filename_prefix}_all_numeric_medians.csv')

    representatives = []
    scores = cycle['scores']
    for cluster in np.unique(labels):
        positions = np.flatnonzero(labels == cluster)
        centroid = scores[positions].mean(axis=0)
        closest = positions[np.argsort(np.linalg.norm(scores[positions] - centroid,
                                                     axis=1))[:3]]
        representatives.append({
            'cluster': int(cluster + 1), 'n': len(positions),
            'centroid_nearest_countries': '; '.join(cohort.loc[closest, ID]),
            'first_five_alphabetically': '; '.join(
                sorted(cohort.loc[positions, ID])[:5]),
        })
    pd.DataFrame(representatives).to_csv(
        OUT / f'{filename_prefix}_representatives.csv', index=False)
    df.to_csv(OUT / f'{filename_prefix}_country_profiles.csv', index=False)
    return medians, pd.DataFrame(representatives)

base_medians,_ = profile(cohort,cycles[0],labels[('baseline','KMeans',5)],'baseline_KMeans_k5')
rev_medians,rev_reps = profile(cohort,cycles[1],labels[('revision','KMeans',3)],'revision_KMeans_k3')
ward_medians,_ = profile(cohort,cycles[1],labels[('revision','Ward',3)],'revision_Ward_k3')
display(comparison.loc[(comparison.cycle=='revision')&(comparison.method=='KMeans')&(comparison.k==3)])
display(rev_medians);display(rev_reps)

## 8.1 Interpret Group Patterns — verify completion zeros

In [ ]:
completion_fields = [c for c in cohort if c.startswith('Completion_Rate_')]
all_six_zero = cohort[completion_fields].eq(0).all(axis=1)
flags = pd.DataFrame({'cluster':labels[('revision','KMeans',3)]+1,
                      'all_six_completion_zero':all_six_zero})
display(flags.groupby('cluster').all_six_completion_zero.agg(['sum','count']))

## 8.2 Visualise Data Models and Patterns — actual outputs

In [ ]:
def plot_selected(cycles, labels_by_key, cohort, medians):
    base = labels_by_key[('baseline', 'KMeans', 5)]
    revised = labels_by_key[('revision', 'KMeans', 3)]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    for ax, labels, title in ((axes[0], base, 'Baseline K-Means k=5'),
                              (axes[1], revised, 'Revision K-Means k=3')):
        counts = np.bincount(labels)
        ax.bar(np.arange(1, len(counts) + 1), counts, color='#37618d')
        for i, n in enumerate(counts, 1):
            ax.text(i, n + 0.5, str(n), ha='center', fontsize=8)
        ax.set(xlabel='Cluster label (arbitrary)', ylabel='Countries / areas',
               title=title, ylim=(0, max(counts) * 1.16))
    save_figure('cluster_sizes.png')

    cycle = cycles[1]
    fig, ax = plt.subplots(figsize=(6.1, 4.5))
    colors = ['#37618d', '#c65d3c', '#549277']
    for cluster in range(3):
        mask = revised == cluster
        ax.scatter(cycle['scores'][mask, 0], cycle['scores'][mask, 1],
                   s=26, alpha=.75, color=colors[cluster],
                   label=f'C{cluster+1} (n={mask.sum()})')
    ax.set(xlabel=f'PC1 ({cycle["pca"].explained_variance_ratio_[0]:.1%})',
           ylabel=f'PC2 ({cycle["pca"].explained_variance_ratio_[1]:.1%})',
           title='Revision k=3, projection of seven-component clustering')
    ax.legend(frameon=False)
    save_figure('revision_pca_projection.png')

    columns = [
        'OOSR_Upper_Secondary_Age_Female',
        'Completion_Rate_Primary_Female',
        'Completion_Rate_Upper_Secondary_Female',
        'Youth_15_24_Literacy_Rate_Female',
        'Gross_Tertiary_Education_Enrollment',
        'Birth_Rate', 'GDP_per_capita_2024',
    ]
    z = (medians[columns] - cohort[columns].median()) / cohort[columns].std()
    z = z.clip(-2, 2)
    fig, ax = plt.subplots(figsize=(10, 2.8))
    im = ax.imshow(z, cmap='RdBu', vmin=-2, vmax=2, aspect='auto')
    ax.set_yticks(range(3), [f'C{i} (n={n})' for i, n in zip(medians.index, medians['n'])])
    ax.set_xticks(range(len(columns)), [
        'Female upper-sec. OOS', 'Female primary completion',
        'Female upper-sec. completion', 'Female youth literacy',
        'Tertiary enrolment', 'Birth rate', 'GDP / capita',
    ], rotation=26, ha='right')
    fig.colorbar(im, ax=ax, fraction=.02, pad=.02,
                 label='Deviation (SD)')
    ax.set_title('Revision k=3: medians of original indicators; see CSV for raw units')
    save_figure('revision_profiles_heatmap.png')

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    for ax, key, scores, name in [
        (axes[0], ('baseline', 'KMeans', 5), cycles[0]['scores'], 'Baseline k=5'),
        (axes[1], ('revision', 'KMeans', 3), cycles[1]['scores'], 'Revision k=3'),
    ]:
        labels = labels_by_key[key]
        sample_silhouettes = silhouette_samples(scores, labels)
        y = 10
        for c in np.unique(labels):
            vals = np.sort(sample_silhouettes[labels == c])
            ax.fill_betweenx(np.arange(y, y + len(vals)), 0, vals,
                             alpha=.75, color=colors[int(c) % len(colors)])
            ax.text(-.13, y + len(vals) / 2, f'C{c+1}', va='center')
            y += len(vals) + 9
        ax.axvline(sample_silhouettes.mean(), linestyle='--', color='black')
        ax.set(xlabel='Silhouette of each country', ylabel='Countries grouped by cluster',
               title=f'{name}: mean {sample_silhouettes.mean():.3f}',
               yticks=[], xlim=(-.22, 1))
    save_figure('silhouette_selected.png')

plot_selected(cycles,labels,cohort,rev_medians)
for filename in ('cluster_sizes.png','revision_pca_projection.png',
                 'revision_profiles_heatmap.png','silhouette_selected.png'):
    display(Image(filename=str(FIG/filename),width=850))

## 8.3 Interpret Results — exploratory use with source limits

In [ ]:
print('All-six-zero completion records:',int(all_six_zero.sum()),'of',len(cohort));print('Source zeros and years require validation.')

## 8.4 Evaluate and Repeat Steps — full C/D robustness passes

B has been fitted and reviewed. C removes GDP, while D excludes 73 records whose six completion fields all equal zero. Refit PCA and both algorithms for k=2..8 in each pass; D is a selected cohort, not an improved replacement.

In [ ]:
def sensitivity(cohort, cycles, labels_by_key):
    selected = labels_by_key[('revision', 'KMeans', 3)]
    alternate = labels_by_key[('revision', 'Ward', 3)]
    prior_k3 = labels_by_key[('baseline', 'KMeans', 3)]
    se = []
    for seed in range(20):
        _, new_labels = fit_candidate(cycles[1]['scores'], 'KMeans', 3, seed)
        se.append(adjusted_rand_score(selected, new_labels))
    no_gdp = labels_by_key[('education_only', 'KMeans', 3)]
    outcome = {
        'revised_kmeans_k3_vs_revised_ward_k3_ARI': float(adjusted_rand_score(selected, alternate)),
        'revised_kmeans_k3_vs_baseline_kmeans_k3_ARI': float(adjusted_rand_score(selected, prior_k3)),
        'revised_kmeans_k3_vs_no_GDP_revised_features_k3_ARI': float(adjusted_rand_score(selected, no_gdp)),
        'revised_kmeans_k3_seed_0_to_19_min_ARI': float(min(se)),
        'revised_kmeans_k3_seed_0_to_19_mean_ARI': float(np.mean(se)),
        'education_only_components_to_90pct': cycles[2]['n_components'],
        'education_only_explained_variance': cycles[2]['explained'],
    }
    completion_fields = [c for c in cohort if c.startswith('Completion_Rate_')]
    literacy_fields = [c for c in cohort if c.startswith('Youth_15_24_Literacy_Rate_')]
    proficiency_fields = [c for c in cohort if 'Proficiency' in c]
    quality_flags = pd.DataFrame({
        ID: cohort[ID], 'cluster': selected + 1,
        'all_six_completion_zero': cohort[completion_fields].eq(0).all(axis=1),
        'both_literacy_zero': cohort[literacy_fields].eq(0).all(axis=1),
        'all_six_proficiency_zero': cohort[proficiency_fields].eq(0).all(axis=1),
    })
    group_flags = quality_flags.groupby('cluster').agg({
        ID: 'count', 'all_six_completion_zero': 'sum',
        'both_literacy_zero': 'sum', 'all_six_proficiency_zero': 'sum',
    }).rename(columns={ID: 'n'})
    group_flags.to_csv(OUT / 'quality_flags_by_cluster.csv')
    # Pass D deliberately tests this nonrandom exclusion. It cannot replace
    # the primary B model without evidence that all recorded zeros are missing.
    keep = ~quality_flags['all_six_completion_zero'].to_numpy()
    complete_labels = labels_by_key[('completion_nonzero', 'KMeans', 2)]
    complete_k3 = labels_by_key[('completion_nonzero', 'KMeans', 3)]
    assert len(complete_labels) == int(keep.sum()) == len(cycles[3]['cohort'])
    outcome.update({
        'all_six_completion_zero_total': int((~keep).sum()),
        'completion_nonzero_sensitivity_n': int(keep.sum()),
        'completion_nonzero_sensitivity_components': cycles[3]['n_components'],
        'completion_nonzero_selected_k': 2,
        'completion_nonzero_selected_k2_silhouette': float(
            silhouette_score(cycles[3]['scores'], complete_labels)),
        'original_vs_completion_nonzero_selected_k2_on_93_ARI': float(
            adjusted_rand_score(selected[keep], complete_labels)),
        'completion_nonzero_sensitivity_k3_silhouette': float(
            silhouette_score(cycles[3]['scores'], complete_k3)),
        'original_vs_completion_nonzero_sensitivity_on_93_ARI': float(
            adjusted_rand_score(selected[keep], complete_k3)),
    })
    with (OUT / 'sensitivity_summary.json').open('w') as f:
        json.dump(outcome, f, indent=2)
    return outcome

remaining = cohort.loc[~all_six_zero].reset_index(drop=True)
assert len(remaining)==93
remaining.to_csv(OUT/'modelling_cohort_93.csv',index=False)
cycles.extend([
 prepare_cycle(cohort,[f for f in revision_columns if f!='log_GDP_2024'],'education_only'),
 prepare_cycle(remaining,revision_columns,'completion_nonzero'),
])
comparison, labels = compare_models(cycles,cohort)
assert len(comparison)==56
for cycle in cycles[2:]:
 for method in ('KMeans','Ward'):
  k = 2 if cycle['name']=='completion_nonzero' else 3
  profile(cycle['cohort'],cycle,labels[(cycle['name'],method,k)],
          f"{cycle['name']}_{method}_k{k}")
profile(remaining,cycles[3],labels[('completion_nonzero','KMeans',3)],
        'completion_nonzero_KMeans_k3')
robust=sensitivity(cohort,cycles,labels)
plot_pca(cycles);plot_comparison(comparison);plot_four_pass_summary(comparison)
display(pd.DataFrame([robust]).T.rename(columns={0:'value'}))
display(Image(filename=str(FIG/'four_pass_summary.png'),width=850))

## 8.5 Four Documented Passes — compare outputs and limits

In [ ]:
selected = comparison.loc[(comparison.method=='KMeans') &
 (((comparison.cycle=='baseline')&(comparison.k==5)) |
  ((comparison.cycle.isin(['revision','education_only']))&(comparison.k==3)) |
  ((comparison.cycle=='completion_nonzero')&(comparison.k==2)))]
display(selected[['cycle','method','k','silhouette','davies_bouldin',
                  'calinski_harabasz','cluster_sizes']])
print('D samples 93 nonrandomly selected records. Review all 56 candidates:',OUT/'model_comparison.csv')
display(Image(filename=str(FIG/'four_pass_pca.png'),width=850))
audit.update({
 'baseline_feature_count':len(baseline_columns),
 'baseline_n_components':cycles[0]['n_components'],
 'baseline_variance_explained':cycles[0]['explained'],
 'revision_feature_count':len(revision_columns),
 'revision_excluded_fields':excluded_fields,
 'revision_n_components':cycles[1]['n_components'],
 'revision_variance_explained':cycles[1]['explained'],
 'four_passes':[{'cycle':c['name'],'records':len(c['cohort']),
                 'feature_count':len(c['columns']),
                 'n_components':c['n_components'],
                 'explained_variance':c['explained']} for c in cycles],
 'package_versions':{'pandas':pd.__version__,'numpy':np.__version__,
                     'scikit_learn':sklearn.__version__,'scipy':scipy.__version__,
                     'matplotlib':matplotlib.__version__},
})
(OUT/'audit_summary.json').write_text(json.dumps(audit,indent=2))

## 下载输出 / Download results

截图应取自运行后的 Colab 代码和输出。报告中的“code and output panels”由真实源代码及保存的运行结果排版生成，不是 Colab 截图。

In [ ]:
from google.colab import files
import shutil
files.download(shutil.make_archive('iteration3_colab_results','zip',
                                     root_dir=str(project_root),base_dir='outputs'))